# GitHub NLP Analysis - AI Coding Assistant Trust & Distrust

**COSC 3047 Assignment 2**

This notebook applies the same text analysis structure used for the YouTube comments to GitHub issue and issue-comment text. It produces repository-level sentiment, topic, and cross-analysis outputs for comparison with the YouTube findings.


## 1. Setup & Imports

In [ ]:

import os
import json
import re
import logging
from pathlib import Path

import nltk
import pandas as pd
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from gensim import corpora
from gensim.models import LdaModel
from wordcloud import WordCloud

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)
logging.getLogger("gensim").setLevel(logging.WARNING)

for resource in ["stopwords", "punkt", "wordnet", "omw-1.4", "punkt_tab"]:
    nltk.download(resource, quiet=True)

print("Imports done.")


## 2. Configuration

### LDA Settings

The GitHub analysis keeps the same topic count, pass count, random seed, and theme seed structure used in the YouTube NLP notebook. This keeps the two platform analyses comparable while still allowing GitHub-specific text cleaning.

### Theme Seeds

The seed words are used only to assign interpretable labels to LDA topics after training. The model itself remains unsupervised.


In [ ]:
def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "collection_notebooks").exists() and (path / "src").exists():
            return path
    for path in [current, *current.parents]:
        nested = path / "social-media-2"
        if (nested / "collection_notebooks").exists() and (nested / "src").exists():
            return nested
    raise FileNotFoundError("Could not find the social-media-2 project root")

PROJECT_ROOT = find_project_root()
INPUT_PATH = PROJECT_ROOT / "data/processed/github/GitHubDiscussionTextFiltered.csv"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/nlp/github"
LDA_DIR = OUTPUT_DIR / "lda_model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError("Run the GitHub collection/cleaning notebook first to create GitHubDiscussionTextFiltered.csv")

df = pd.read_csv(INPUT_PATH)
df = df[df["textClean"].notna()].copy()
df = df[df["textClean"].astype(str).str.strip() != ""].copy()
df["createdAt"] = pd.to_datetime(df["createdAt"], errors="coerce")

print(f"Loaded {len(df):,} GitHub text records")
print(f"Repositories: {df['repository'].nunique():,}")
print(f"Issues: {df['issueKey'].nunique():,}")
print(f"Authors: {df['author'].nunique():,}")
print(f"Date range: {df['createdAt'].min()} to {df['createdAt'].max()}")
df.head(3)


In [ ]:
NUM_TOPICS = 6
LDA_PASSES = 15
LDA_RANDOM_SEED = 42

THEME_SEEDS = {
    "productivity": ["faster", "speed", "time", "efficient", "workflow",
                     "automate", "quick", "save", "boilerplate", "autocomplete",
                     "suggest", "generate", "write", "complete"],
    "reliability": ["bug", "error", "wrong", "incorrect", "broken",
                    "unreliable", "hallucinate", "mistake", "garbage",
                    "fix", "issue", "fail", "trash", "useless", "test",
                    "crash", "exception", "timeout"],
    "security": ["security", "privacy", "data", "leak", "vulnerable",
                 "license", "safe", "risk", "expose", "telemetry",
                 "permission", "token", "credential"],
    "cost": ["price", "expensive", "cheap", "cost", "subscription",
             "free", "worth", "money", "pay", "pricing", "plan", "open"],
    "code_ownership": ["copyright", "ownership", "training", "stolen",
                       "plagiarism", "license", "legal", "intellectual",
                       "property", "scraping"],
    "job_displacement": ["job", "replace", "replacing", "replaced", "unemployed",
                         "fired", "layoff", "obsolete", "human", "engineer",
                         "developer", "programmer", "junior", "hire", "career"],
}

EXTRA_STOPWORDS = {
    "ai", "code", "coding", "use", "using", "used", "tool", "like", "just",
    "get", "got", "one", "would", "could", "really", "think", "know", "also",
    "github", "copilot", "cursor", "claude", "codex", "vscode", "openai",
    "anthropic", "anthropics", "microsoft", "comment", "issue", "repo",
    "repository", "please", "thanks", "thank", "hi", "hello", "see", "say",
    "said", "make", "made", "want", "need", "new", "go", "going", "still",
    "even", "well", "thing", "things", "something", "already", "actually",
    "version", "latest", "current", "expected", "actual", "behavior",
    "reproduce", "step", "steps", "environment", "output", "log", "logs",
    "file", "line", "window", "windows", "linux", "mac", "node", "npm",
    "extension", "chat", "model", "app", "cli", "terminal", "user"
}

stop_words = set(stopwords.words("english")) | EXTRA_STOPWORDS
print(f"Total stopwords: {len(stop_words):,}")


## 3. Text Preprocessing

GitHub issues often contain Markdown, code snippets, stack traces, version numbers, and issue references. The collection notebook already creates a cleaned text field; this stage applies the additional token-level preprocessing needed for topic modelling.


In [ ]:
def preprocess_text(text: str, stop_words: set, lemmatizer: WordNetLemmatizer) -> list:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@[a-z0-9_-]+", " ", text)
    text = re.sub(r"#[0-9]+", " ", text)
    text = re.sub(r"\b[a-f0-9]{8,}\b", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) >= 3
    ]
    return tokens

print("preprocess_text defined.")


## 4. Sentiment Analysis (VADER)

The same VADER thresholds used in the YouTube analysis are applied here: positive scores are `>= 0.05`, negative scores are `<= -0.05`, and all remaining records are neutral. GitHub issues are naturally weighted toward problem reports, so the results should be interpreted as discussion tone rather than a direct product rating.


In [ ]:
def run_sentiment_analysis(df: pd.DataFrame) -> pd.DataFrame:
    analyser = SentimentIntensityAnalyzer()
    results = []
    for text in df["textClean"].fillna(""):
        scores = analyser.polarity_scores(str(text))
        compound = scores["compound"]
        label = "neutral"
        if compound >= 0.05:
            label = "positive"
        elif compound <= -0.05:
            label = "negative"
        results.append({
            "sentiment_compound": compound,
            "sentiment_pos": scores["pos"],
            "sentiment_neg": scores["neg"],
            "sentiment_neu": scores["neu"],
            "sentiment_label": label,
        })
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)

print("run_sentiment_analysis defined.")


In [ ]:
sent_df = run_sentiment_analysis(df)

print("Sentiment distribution:")
print(sent_df["sentiment_label"].value_counts())
print(f"\nAverage compound score: {sent_df['sentiment_compound'].mean():.4f}")
sent_df[["repository", "recordType", "textClean", "sentiment_compound", "sentiment_label"]].head(5)


### Sentiment Visualisations

In [ ]:
COLORS = {"positive": "#4CAF50", "neutral": "#FF9800", "negative": "#F44336"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Sentiment Analysis - GitHub Issues and Comments", fontsize=13, fontweight="bold")

counts = sent_df["sentiment_label"].value_counts()
axes[0].bar(counts.index, counts.values, color=[COLORS[s] for s in counts.index], edgecolor="white")
axes[0].set_title("Overall Sentiment Distribution")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Number of Records")
for i, value in enumerate(counts.values):
    axes[0].text(i, value + max(counts.values) * 0.02, f"{value:,}", ha="center", fontweight="bold", fontsize=9)

avg_repo = sent_df.groupby("repository")["sentiment_compound"].mean().sort_values()
bar_colors = ["#F44336" if value < 0 else "#4CAF50" for value in avg_repo.values]
axes[1].barh(avg_repo.index, avg_repo.values, color=bar_colors, edgecolor="white")
axes[1].axvline(x=0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("Avg Sentiment by Repository")
axes[1].set_xlabel("Avg Compound Score")

record_sent = sent_df.groupby(["recordType", "sentiment_label"]).size().unstack(fill_value=0)
record_sent.plot(kind="bar", ax=axes[2], color=[COLORS[c] for c in record_sent.columns], edgecolor="white")
axes[2].set_title("Sentiment by Record Type")
axes[2].set_xlabel("Record Type")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=0)
axes[2].legend(title="Sentiment")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "github_sentiment_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved github_sentiment_overview.png")


## 5. Topic Modelling (LDA)

LDA is trained on the GitHub discussion text using the same broad topic structure as the YouTube notebook. The resulting topics are mapped back to the shared research themes using the seed terms above.


In [ ]:
def run_topic_modelling(sent_df: pd.DataFrame, stop_words: set):
    lemmatizer = WordNetLemmatizer()
    log.info("Preprocessing text for LDA...")
    texts = [
        preprocess_text(str(text), stop_words, lemmatizer)
        for text in sent_df["textClean"].fillna("")
    ]
    texts = [tokens for tokens in texts if len(tokens) > 0]
    log.info(f"  {len(texts):,} non-empty documents after preprocessing")

    dictionary = corpora.Dictionary(texts)
    dictionary.filter_extremes(no_below=3, no_above=0.7)
    log.info(f"  Vocabulary size after filtering: {len(dictionary):,} terms")

    corpus = [dictionary.doc2bow(text) for text in texts]

    log.info(f"Training LDA ({NUM_TOPICS} topics, {LDA_PASSES} passes)...")
    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=NUM_TOPICS,
        passes=LDA_PASSES,
        random_state=LDA_RANDOM_SEED,
        alpha="auto",
        eta="auto",
    )
    log.info("LDA training complete.")

    LDA_DIR.mkdir(parents=True, exist_ok=True)
    lda_model.save(str(LDA_DIR / "lda.model"))
    dictionary.save(str(LDA_DIR / "dictionary.gensim"))
    log.info(f"Model saved to {LDA_DIR}")

    return lda_model, dictionary


def map_topics_to_themes(lda_model: LdaModel, num_words: int = 15):
    topic_map = {}
    topic_terms = {}
    for topic_id in range(NUM_TOPICS):
        terms = [word for word, _ in lda_model.show_topic(topic_id, topn=num_words)]
        topic_terms[topic_id] = terms
        scores = {
            theme: sum(1 for term in terms if term in seeds)
            for theme, seeds in THEME_SEEDS.items()
        }
        best_theme = max(scores, key=scores.get)
        topic_map[topic_id] = best_theme if scores[best_theme] > 0 else "other"
        log.info(f"  Topic {topic_id} -> '{topic_map[topic_id]}' | top terms: {', '.join(terms[:8])}")
    return topic_map, topic_terms


def assign_topics_to_records(sent_df: pd.DataFrame, lda_model: LdaModel,
                             dictionary: corpora.Dictionary, topic_map: dict,
                             stop_words: set) -> pd.DataFrame:
    lemmatizer = WordNetLemmatizer()
    topic_ids, topic_labels, topic_scores = [], [], []
    log.info("Assigning topics to records...")
    for text in sent_df["textClean"].fillna(""):
        tokens = preprocess_text(str(text), stop_words, lemmatizer)
        bow = dictionary.doc2bow(tokens)
        if not bow:
            topic_ids.append(None)
            topic_labels.append("unknown")
            topic_scores.append(0.0)
            continue
        topic_dist = lda_model.get_document_topics(bow)
        if not topic_dist:
            topic_ids.append(None)
            topic_labels.append("unknown")
            topic_scores.append(0.0)
            continue
        dominant = max(topic_dist, key=lambda item: item[1])
        topic_ids.append(int(dominant[0]))
        topic_labels.append(topic_map.get(dominant[0], "other"))
        topic_scores.append(round(float(dominant[1]), 4))
    log.info("Topic assignment complete.")

    result = sent_df.copy()
    result["dominant_topic_id"] = topic_ids
    result["topic_label"] = topic_labels
    result["topic_score"] = topic_scores
    return result

print("LDA functions defined.")


### Train or Load LDA Model

In [ ]:
if (LDA_DIR / "lda.model").exists() and (LDA_DIR / "dictionary.gensim").exists():
    log.info("Found existing GitHub LDA model - loading instead of retraining.")
    log.info("Delete data/processed/nlp/github/lda_model/ to force retrain.")
    lda_model = LdaModel.load(str(LDA_DIR / "lda.model"))
    dictionary = corpora.Dictionary.load(str(LDA_DIR / "dictionary.gensim"))
else:
    log.info("No saved model found - training from scratch.")
    lda_model, dictionary = run_topic_modelling(sent_df, stop_words)

log.info("Mapping topics to research themes...")
topic_map, topic_terms = map_topics_to_themes(lda_model)

topic_terms_output = {
    str(topic_id): {"theme": topic_map[topic_id], "top_terms": terms}
    for topic_id, terms in topic_terms.items()
}
with open(OUTPUT_DIR / "github_topic_model_terms.json", "w") as f:
    json.dump(topic_terms_output, f, indent=2)
print("Saved github_topic_model_terms.json")


### Assign Topics to Records

In [ ]:
topic_df = assign_topics_to_records(sent_df, lda_model, dictionary, topic_map, stop_words)

print("Topic distribution:")
print(topic_df["topic_label"].value_counts())

topic_df.to_csv(OUTPUT_DIR / "github_text_with_topics.csv", index=False)
print(f"\nSaved github_text_with_topics.csv ({len(topic_df):,} rows)")


### Topic Visualisations

In [ ]:
TOPIC_COLORS = {
    "productivity": "#2196F3",
    "reliability": "#F44336",
    "security": "#FF9800",
    "cost": "#9C27B0",
    "code_ownership": "#009688",
    "job_displacement": "#795548",
    "other": "#9E9E9E",
    "unknown": "#BDBDBD",
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("LDA Topic Modelling - GitHub Discussions", fontsize=13, fontweight="bold")

topic_counts = topic_df["topic_label"].value_counts()
bar_colors = [TOPIC_COLORS.get(topic, "#9E9E9E") for topic in topic_counts.index]
axes[0].bar(topic_counts.index, topic_counts.values, color=bar_colors, edgecolor="white")
axes[0].set_title("Topic Distribution Across GitHub Records")
axes[0].set_xlabel("Theme")
axes[0].set_ylabel("Number of Records")
axes[0].tick_params(axis="x", rotation=30)
for i, value in enumerate(topic_counts.values):
    axes[0].text(i, value + max(topic_counts.values) * 0.02, f"{value:,}", ha="center", fontsize=9, fontweight="bold")

terms_data = {f"Topic {topic_id}\n({topic_map[topic_id]})": topic_terms[topic_id][:8] for topic_id in range(NUM_TOPICS)}
terms_df = pd.DataFrame(terms_data)
axes[1].axis("off")
table = axes[1].table(cellText=terms_df.values, colLabels=terms_df.columns,
                      cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1.2, 1.5)
axes[1].set_title("Top 8 Terms per LDA Topic", fontweight="bold", pad=20)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "github_topic_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved github_topic_overview.png")


## 6. Cross-Analysis - Sentiment x Topic

This section connects topic labels with sentiment scores so GitHub support friction can be compared with the YouTube public-comment results.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Sentiment x Topic Cross-Analysis - GitHub", fontsize=13, fontweight="bold")

valid_topics = [topic for topic in THEME_SEEDS if topic in topic_df["topic_label"].unique()]
topic_sent = topic_df[topic_df["topic_label"].isin(valid_topics)].groupby(
    ["topic_label", "sentiment_label"]
).size().unstack(fill_value=0)
topic_sent.plot(kind="bar", ax=axes[0],
                color=[COLORS.get(label, "#9E9E9E") for label in topic_sent.columns],
                edgecolor="white")
axes[0].set_title("Sentiment Distribution per Research Theme")
axes[0].set_xlabel("Theme")
axes[0].set_ylabel("Record Count")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(title="Sentiment")

avg_by_topic = topic_df[topic_df["topic_label"].isin(valid_topics)].groupby(
    "topic_label"
)["sentiment_compound"].mean().sort_values()
bar_colors = ["#F44336" if value < 0 else "#4CAF50" for value in avg_by_topic.values]
axes[1].barh(avg_by_topic.index, avg_by_topic.values, color=bar_colors, edgecolor="white")
axes[1].axvline(x=0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("Avg Sentiment Score per Research Theme")
axes[1].set_xlabel("Avg Compound Score")
for i, value in enumerate(avg_by_topic.values):
    axes[1].text(value + 0.002, i, f"{value:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "github_sentiment_by_topic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved github_sentiment_by_topic.png")


### Positive vs Negative Word Clouds

These word clouds show the terms most visible in positive and negative GitHub discussion records after applying the same stopword approach used in the topic model.


In [ ]:
positive_text = " ".join(
    topic_df[topic_df["sentiment_label"] == "positive"]["textClean"].fillna("")
)
negative_text = " ".join(
    topic_df[topic_df["sentiment_label"] == "negative"]["textClean"].fillna("")
)

wc_stopwords = stop_words | {
    "will", "can", "may", "might", "back", "look", "let", "try", "trying",
    "based", "without", "within", "around", "since", "every", "anything"
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor("#0d1117")

wc_pos = WordCloud(
    width=800, height=500,
    background_color="#0d1117",
    colormap="Greens",
    stopwords=wc_stopwords,
    max_words=80,
    prefer_horizontal=0.85,
    collocations=False,
).generate(positive_text)

axes[0].imshow(wc_pos, interpolation="bilinear")
axes[0].axis("off")
axes[0].set_title("Trust Signals - Positive GitHub Records", fontsize=14, fontweight="bold", color="white", pad=15)
axes[0].set_facecolor("#0d1117")

wc_neg = WordCloud(
    width=800, height=500,
    background_color="#0d1117",
    colormap="Reds",
    stopwords=wc_stopwords,
    max_words=80,
    prefer_horizontal=0.85,
    collocations=False,
).generate(negative_text)

axes[1].imshow(wc_neg, interpolation="bilinear")
axes[1].axis("off")
axes[1].set_title("Distrust Signals - Negative GitHub Records", fontsize=14, fontweight="bold", color="white", pad=15)
axes[1].set_facecolor("#0d1117")

plt.suptitle("GitHub Discussion Tone - Trust vs Distrust Signals", fontsize=15, fontweight="bold", color="white", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "github_wordcloud_trust_distrust.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("Saved github_wordcloud_trust_distrust.png")


### Topic Distribution by Repository

Each cell shows the percentage of records from a repository assigned to each theme. Percentages are used because the repositories have different numbers of collected comments and issues.


In [ ]:
pivot = (
    topic_df.groupby(["repository", "topic_label"])
    .size()
    .unstack(fill_value=0)
)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

theme_cols = [
    col for col in ["productivity", "reliability", "security", "cost", "code_ownership", "job_displacement"]
    if col in pivot_pct.columns
]
pivot_pct = pivot_pct[theme_cols]

fig, ax = plt.subplots(figsize=(12, 5))
vmax = max(80, pivot_pct.max().max() if not pivot_pct.empty else 80)
image = ax.imshow(pivot_pct.values, cmap="YlOrRd", vmin=0, vmax=vmax, aspect="auto")
cbar = fig.colorbar(image, ax=ax, shrink=0.8)
cbar.set_label("% of repository records")

ax.set_xticks(range(len(pivot_pct.columns)))
ax.set_xticklabels(pivot_pct.columns, rotation=30, ha="right")
ax.set_yticks(range(len(pivot_pct.index)))
ax.set_yticklabels(pivot_pct.index)

for row_index in range(pivot_pct.shape[0]):
    for col_index in range(pivot_pct.shape[1]):
        value = pivot_pct.iloc[row_index, col_index]
        ax.text(col_index, row_index, f"{value:.1f}", ha="center", va="center", fontsize=9)

ax.set_title("Topic Distribution by GitHub Repository (% of Records)", fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Research Theme", fontsize=11)
ax.set_ylabel("Repository", fontsize=11)
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "github_topic_repository_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved github_topic_repository_heatmap.png")


## 7. Summary Statistics

In [ ]:
summary = {
    "total_records": int(len(topic_df)),
    "repositories": int(topic_df["repository"].nunique()),
    "issues": int(topic_df["issueKey"].nunique()),
    "authors": int(topic_df["author"].nunique()),
    "date_min": str(topic_df["createdAt"].min()),
    "date_max": str(topic_df["createdAt"].max()),
    "overall_sentiment": topic_df["sentiment_label"].value_counts().to_dict(),
    "overall_topics": topic_df["topic_label"].value_counts().to_dict(),
    "records_by_repository": topic_df["repository"].value_counts().to_dict(),
    "records_by_type": topic_df["recordType"].value_counts().to_dict(),
    "avg_sentiment_by_repository": (
        topic_df.groupby("repository")["sentiment_compound"].mean().round(4).to_dict()
    ),
    "sentiment_by_topic": (
        topic_df.groupby(["topic_label", "sentiment_label"])
        .size().unstack(fill_value=0).to_dict()
    ),
    "avg_sentiment_by_topic": (
        topic_df.groupby("topic_label")["sentiment_compound"].mean().round(4).to_dict()
    ),
    "topic_by_repository": (
        topic_df.groupby(["repository", "topic_label"])
        .size().unstack(fill_value=0).to_dict()
    ),
}

with open(OUTPUT_DIR / "github_summary_stats.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 50)
print("GitHub NLP Pipeline Complete")
print(f"  Total records analysed: {summary['total_records']:,}")
print(f"  Repositories:           {summary['repositories']:,}")
print(f"  Issues:                 {summary['issues']:,}")
print(f"  Sentiment:              {summary['overall_sentiment']}")
print(f"  Topics:                 {summary['overall_topics']}")
print("  Avg sentiment by repository:")
for repository, score in summary["avg_sentiment_by_repository"].items():
    label = "positive" if score > 0.05 else "negative" if score < -0.05 else "neutral"
    print(f"    {repository:40s}: {score:+.4f} ({label})")
print("=" * 50)


## 8. Outputs

All outputs are saved to `data/processed/nlp/github/`:

| File | Description |
|---|---|
| `github_text_with_topics.csv` | GitHub issues/comments with VADER scores and topic labels |
| `github_topic_model_terms.json` | Top terms per LDA topic and mapped research theme |
| `github_summary_stats.json` | Aggregate breakdowns by repository, record type, topic, and sentiment |
| `github_sentiment_overview.png` | Sentiment distribution charts |
| `github_topic_overview.png` | Topic distribution and top terms table |
| `github_sentiment_by_topic.png` | Sentiment per research theme |
| `github_wordcloud_trust_distrust.png` | Positive and negative GitHub word clouds |
| `github_topic_repository_heatmap.png` | Topic distribution by repository |
